# 🔍 02 — Embeddings & Retrieval Quality Analysis

This notebook validates the output of the embedding and retrieval pipeline (Week 2).
Use it to:
- Visualise the FAISS index with UMAP (2D projection)
- Test retrieval quality with sample queries
- Compare FAISS-only vs FAISS + Cohere rerank results
- Spot retrieval failures before building the full RAG pipeline

**Run this after `embeddings.py`**

In [2]:
import os
import pickle
import sys
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Allow imports from project root
sys.path.insert(0, str(Path('..').resolve()))
load_dotenv('../.env')

from src.retrieval.retriever import load_index, retrieve, embed_query

INDEX_DIR = Path('../data/index')
COLORS = {'tesla': '#E31937', 'apple': '#555555', 'microsoft': '#00A4EF'}

c:\Users\pablo\Desktop\financial-rag-assistant\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load index

In [3]:
index, metadata = load_index()
df_meta = pd.DataFrame(metadata)

print(f'Total vectors : {index.ntotal:,}')
print(f'Embedding dim : {index.d}')
print()
print(df_meta.groupby('company').size().rename('chunks').to_string())

FileNotFoundError: Index not found. Run src/embeddings/embeddings.py first.

## 2. UMAP visualisation — do embeddings cluster by company?

In [ ]:
# pip install umap-learn  (only needed for this notebook)
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('Install umap-learn to see the 2D projection: pip install umap-learn')

if HAS_UMAP:
    # Reconstruct all vectors from the flat index
    all_vectors = faiss.rev_swig_ptr(index.get_xb(), index.ntotal * index.d)
    all_vectors = np.array(all_vectors).reshape(index.ntotal, index.d)

    # Sample max 3000 points for speed
    sample_size = min(3000, index.ntotal)
    sample_idx = np.random.choice(index.ntotal, sample_size, replace=False)
    sample_vecs = all_vectors[sample_idx]
    sample_meta = [metadata[i] for i in sample_idx]

    print(f'Running UMAP on {sample_size} vectors…')
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    projected = reducer.fit_transform(sample_vecs)

    fig, ax = plt.subplots(figsize=(10, 7))
    for company, color in COLORS.items():
        mask = [m['company'] == company for m in sample_meta]
        pts = projected[mask]
        ax.scatter(pts[:, 0], pts[:, 1], c=color, label=company.capitalize(),
                   alpha=0.4, s=4)

    ax.set_title('UMAP projection of 10-K embeddings (BGE-large-en-v1.5)', fontsize=13)
    ax.legend(markerscale=3)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../data/index/umap_projection.png', dpi=150)
    plt.show()
    print('Saved to data/index/umap_projection.png')

## 3. Manual retrieval tests

In [ ]:
TEST_QUERIES = [
    ('What are the main risk factors for Tesla?',        'tesla'),
    ('How does Apple describe Services segment revenue?','apple'),
    ('What is Microsoft Azure revenue growth?',          'microsoft'),
    ('Compare risk factors across all three companies',  None),
]

for query, company_filter in TEST_QUERIES:
    print(f'\n{"─"*65}')
    print(f'  Query  : {query}')
    print(f'  Filter : {company_filter or "all companies"}')
    print(f'{"─"*65}')

    results = retrieve(query, index, metadata, company_filter=company_filter)

    for i, chunk in enumerate(results, 1):
        score = chunk.get('rerank_score', chunk.get('faiss_score', 0))
        print(f'  [{i}] {chunk["company"].upper():12s} | '
              f'p{chunk["start_page"]}–{chunk["end_page"]} | '
              f'score={score:.4f}')
        print(f'       {chunk["text"][:180]}…')

## 4. FAISS-only vs. Reranked — side-by-side comparison

In [ ]:
from src.retrieval.retriever import faiss_search, rerank

COMPARISON_QUERY = 'What macroeconomic risks does Tesla face?'

faiss_results  = faiss_search(COMPARISON_QUERY, index, metadata, top_k=5)
rerank_results = rerank(COMPARISON_QUERY, faiss_search(COMPARISON_QUERY, index, metadata, top_k=20))

print(f'Query: "{COMPARISON_QUERY}"\n')

print('─── FAISS only (top 5) ──────────────────────────────────')
for i, c in enumerate(faiss_results, 1):
    print(f'  [{i}] {c["company"].upper():12s} | score={c["faiss_score"]:.4f} | {c["text"][:120]}…')

print('\n─── After Cohere rerank (top 5) ─────────────────────────')
for i, c in enumerate(rerank_results, 1):
    print(f'  [{i}] {c["company"].upper():12s} | score={c["rerank_score"]:.4f} | {c["text"][:120]}…')

## ✅ Checklist before building the RAG pipeline

- [ ] UMAP plot shows company clusters (proves embeddings capture company identity)
- [ ] Single-company queries return only chunks from the right company
- [ ] Reranked results are visibly more relevant than FAISS-only results
- [ ] No query returns 0 results
- [ ] Top-1 result always looks semantically related to the query

If everything looks good → run `python src/pipeline/pipeline.py`